# Read CSV & Parquet from MinIO, Aggregate, and Save Back

This notebook shows you how to:

1. **Read CSV and Parquet files directly from MinIO** — without downloading them to your computer
2. **Load them into pandas DataFrames** for analysis
3. **Perform aggregations** (group-by, sum, mean, etc.)
4. **Save the results as Parquet** into a different MinIO bucket

## How does it work without downloading?

We use the library **s3fs**, which lets pandas talk to any S3-compatible storage (like MinIO)
as if it were a local file system. The data is **streamed** in memory — no temporary file on disk.

## Prerequisites

- Python packages: `pandas`, `s3fs`, `pyarrow` (for Parquet support)
- Environment variables for MinIO connection:
  - `MINIO_ENDPOINT` — server address (e.g. `minio.example.com:9000`)
  - `MINIO_ACCESS_KEY` — your access key
  - `MINIO_SECRET_KEY` — your secret key
  - `MINIO_SECURE` — optional, `true` for HTTPS (defaults to `false`)
- A source bucket containing CSV and/or Parquet files
- A destination bucket where results will be saved (it must already exist)

## Step 1 — Install required packages

- `pandas` — the main data analysis library
- `s3fs` — allows pandas to read/write directly from S3-compatible storage
- `pyarrow` — the engine that reads and writes Parquet files

In [ ]:
# Install the required libraries
!pip install pandas s3fs pyarrow --quiet

## Step 2 — Import libraries and read environment variables

In [ ]:
import os
import pandas as pd  # The main library for tabular data
import s3fs           # Lets pandas read/write to S3-compatible storage

# --- Read connection settings from environment variables ---
# Using environment variables keeps secrets OUT of your code.

endpoint   = os.environ["MINIO_ENDPOINT"]    # e.g. "minio.example.com:9000"
access_key = os.environ["MINIO_ACCESS_KEY"]   # your access key
secret_key = os.environ["MINIO_SECRET_KEY"]   # your secret key
secure     = os.environ.get("MINIO_SECURE", "false").lower() == "true"

print(f"MinIO endpoint : {endpoint}")
print(f"Secure (HTTPS) : {secure}")

## Step 3 — Create the S3 filesystem connection

The `s3fs.S3FileSystem` object is a "virtual file system" that points to your MinIO server.
Once created, pandas can use it to open files on MinIO as easily as local files.

In [ ]:
# Build the endpoint URL that s3fs expects
protocol = "https" if secure else "http"
endpoint_url = f"{protocol}://{endpoint}"

# Create the S3-compatible filesystem object
fs = s3fs.S3FileSystem(
    key=access_key,              # access key (like a username)
    secret=secret_key,           # secret key (like a password)
    endpoint_url=endpoint_url,   # where MinIO is running
    use_ssl=secure,              # match HTTPS setting
)

print(f"Connected to {endpoint_url}")

## Step 4 — Define source and destination buckets

Change these variables to match your setup:
- `SOURCE_BUCKET` — the bucket where your raw CSV/Parquet files live
- `DEST_BUCKET` — the bucket where aggregated results will be saved

In [ ]:
# ---- CHANGE THESE to match your MinIO setup ----
SOURCE_BUCKET = "raw-data"       # bucket containing source files
DEST_BUCKET   = "results"        # bucket where results will be saved

# File names inside the source bucket
CSV_FILE     = "sales.csv"       # a CSV file to read
PARQUET_FILE = "products.parquet" # a Parquet file to read

## Step 5 — Read a CSV file directly from MinIO

We use `fs.open()` to open the file on MinIO as a stream, then pass it to `pd.read_csv()`.
The file is **never saved to disk** — it goes straight into a DataFrame in memory.

In [ ]:
# Build the full path: "bucket-name/file-name"
csv_path = f"{SOURCE_BUCKET}/{CSV_FILE}"

# Open the file on MinIO and read it into a DataFrame
# 'rb' means "read in binary mode" — required for remote files
with fs.open(csv_path, "rb") as f:
    df_csv = pd.read_csv(f)

print(f"CSV loaded: {len(df_csv)} rows, {len(df_csv.columns)} columns")
df_csv.head()  # show the first 5 rows

## Step 6 — Read a Parquet file directly from MinIO

Same approach — open the remote file and read it with `pd.read_parquet()`.
Parquet is a columnar format: it is faster and smaller than CSV for large datasets.

In [ ]:
parquet_path = f"{SOURCE_BUCKET}/{PARQUET_FILE}"

with fs.open(parquet_path, "rb") as f:
    df_parquet = pd.read_parquet(f)

print(f"Parquet loaded: {len(df_parquet)} rows, {len(df_parquet.columns)} columns")
df_parquet.head()

## Step 7 — Perform aggregations

Now that the data is loaded, let's do some analysis.
Below are common aggregation examples — **adapt the column names to your data**.

### What is an aggregation?

An aggregation **summarises many rows into fewer rows** by grouping.
For example: "total sales per product category" groups all rows by category
and sums up the sales column.

In [ ]:
# =============================================
# Example: aggregate the CSV data
# Adapt column names to match YOUR data!
# =============================================

# Suppose df_csv has columns: "category", "amount", "quantity"
#
# groupby("category") = group rows that share the same category
# .agg(...)           = compute summary statistics per group

agg_csv = df_csv.groupby("category").agg(
    total_amount=("amount", "sum"),      # sum of amounts per category
    mean_amount=("amount", "mean"),       # average amount per category
    total_quantity=("quantity", "sum"),    # sum of quantities per category
    row_count=("amount", "count"),         # number of rows per category
).reset_index()  # turn the group labels back into a normal column

print(f"Aggregated CSV: {len(agg_csv)} rows")
agg_csv

In [ ]:
# =============================================
# Example: aggregate the Parquet data
# Adapt column names to match YOUR data!
# =============================================

# Suppose df_parquet has columns: "region", "price", "units_sold"

agg_parquet = df_parquet.groupby("region").agg(
    total_revenue=("price", "sum"),       # total revenue per region
    avg_price=("price", "mean"),           # average price per region
    total_units=("units_sold", "sum"),     # total units sold per region
).reset_index()

print(f"Aggregated Parquet: {len(agg_parquet)} rows")
agg_parquet

## Step 8 — Save results as Parquet to a different MinIO bucket

We write each aggregated DataFrame back to MinIO in **Parquet format**.
Again, no temporary file on disk — data streams directly from memory to MinIO.

In [ ]:
# Save aggregated CSV results
output_path_csv = f"{DEST_BUCKET}/agg_sales.parquet"

with fs.open(output_path_csv, "wb") as f:   # 'wb' = write in binary mode
    agg_csv.to_parquet(f, index=False)        # index=False avoids saving row numbers

print(f"Saved: {output_path_csv}")

# Save aggregated Parquet results
output_path_parquet = f"{DEST_BUCKET}/agg_products.parquet"

with fs.open(output_path_parquet, "wb") as f:
    agg_parquet.to_parquet(f, index=False)

print(f"Saved: {output_path_parquet}")

## Step 9 — Verify the files were saved

Let's list the contents of the destination bucket to confirm our files are there.

In [ ]:
# List all files in the destination bucket
files = fs.ls(DEST_BUCKET, detail=True)

print(f"Files in '{DEST_BUCKET}':\n")
for f in files:
    size_kb = f["size"] / 1024
    print(f"  {f['Key']:<50} {size_kb:>8.1f} KB")

## Summary

In this notebook you learned how to:

1. **Connect to MinIO** using `s3fs` and environment variables
2. **Read CSV files** directly from MinIO into a DataFrame (no download)
3. **Read Parquet files** directly from MinIO into a DataFrame (no download)
4. **Aggregate data** with `groupby()` and `.agg()`
5. **Save results as Parquet** to a different MinIO bucket (no temporary file)

### Key libraries

| Library    | Role                                          |
|------------|-----------------------------------------------|
| `pandas`   | Load, transform, and analyse tabular data     |
| `s3fs`     | Read/write to S3-compatible storage (MinIO)   |
| `pyarrow`  | Read and write Parquet files                  |